# Week 2 Day 3 — LangGraph
## Stateful, Multi-Step & Cyclical Agent Workflows

This notebook completes all five tasks:

1. Graph concepts and State design
2. Linear graph
3. Conditional edges and self-correction cycle
4. Human-in-the-loop interrupt
5. Persistence and debugging

The workflow is a research assistant:

**plan → retrieve → generate → critique → human approval → format**

If the critique score is low, the graph loops back to **generate**. A retry limit prevents an infinite loop.


# Task 1 — Graph Concepts & State Design

## LangGraph Core Building Blocks

### 1. StateGraph
`StateGraph` creates the workflow graph and connects nodes using edges.

### 2. Nodes
Nodes are Python functions. Each node receives the current state and returns updates.

### 3. Edges
Edges define the normal path from one node to another.

### 4. Conditional Edges
Conditional edges choose the next node based on the current state.

### 5. Shared State
The State object contains the information shared by all nodes.

## Graph Design

```text
START
  |
  v
PLAN
  |
  v
RETRIEVE
  |
  v
GENERATE <---------+
  |                |
  v                |
CRITIQUE ----------+
  |
  | good score
  v
HUMAN APPROVAL
  |          |
approved   rejected
  |          |
  v          v
FORMAT      END
  |
  v
 END
```


## State Schema

We use `TypedDict`.

| Field | Purpose |
|---|---|
| question | User question |
| plan | Research plan |
| evidence | Retrieved evidence |
| draft | Generated answer |
| critique | Critic feedback |
| score | Quality score |
| retries | Number of critique passes |
| max_retries | Maximum retry limit |
| approval | Human decision |
| final_answer | Final formatted response |
| history_log | Execution log |


In [1]:
%pip install -U langgraph langchain-core --break-system-packages


Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
class ResearchState(TypedDict, total=False):
    question: str
    plan: List[str]
    evidence: List[str]
    draft: str
    critique: str
    score: int
    retries: int
    max_retries: int
    approval: Optional[str]
    final_answer: str
    history_log: List[str]

print("State schema created successfully.")

State schema created successfully.


# Task 2 — Build a Linear Graph

The first graph contains four nodes:

**plan → retrieve → generate → format**

The graph is deliberately simple so we can verify state updates before adding cycles and interrupts.


In [4]:
def plan_node(state):
    question = state["question"]

    plan = [
        f"Understand the question: {question}",
        "Collect relevant information",
        "Generate an answer",
        "Check the quality of the answer"
    ]

    return {
        "plan": plan,
        "history_log": state.get("history_log", []) + [
            "plan node executed"
        ]
    }


def retrieve_node(state):
    evidence = [
        "LangGraph builds stateful workflows.",
        "Nodes perform individual workflow steps.",
        "Edges control transitions.",
        "Conditional edges support branching and loops."
    ]

    return {
        "evidence": evidence,
        "history_log": state.get("history_log", []) + [
            "retrieve node executed"
        ]
    }


def generate_node(state):
    question = state["question"]
    evidence = state.get("evidence", [])

    draft = (
        f"Question: {question}\n\n"
        "LangGraph allows developers to build stateful agent workflows "
        "using nodes and edges. It provides explicit control over "
        "branching, loops, human approval, and persistence.\n\n"
        f"Evidence used: {len(evidence)} items."
    )

    return {
        "draft": draft,
        "history_log": state.get("history_log", []) + [
            "generate node executed"
        ]
    }


def format_node(state):
    final_answer = (
        "FINAL ANSWER\n"
        "============\n\n"
        + state["draft"]
    )

    return {
        "final_answer": final_answer,
        "history_log": state.get("history_log", []) + [
            "format node executed"
        ]
    }

print("Four linear nodes created.")

Four linear nodes created.


In [5]:
linear_builder = StateGraph(ResearchState)

linear_builder.add_node("plan", plan_node)
linear_builder.add_node("retrieve", retrieve_node)
linear_builder.add_node("generate", generate_node)
linear_builder.add_node("format", format_node)

linear_builder.add_edge(START, "plan")
linear_builder.add_edge("plan", "retrieve")
linear_builder.add_edge("retrieve", "generate")
linear_builder.add_edge("generate", "format")
linear_builder.add_edge("format", END)

linear_graph = linear_builder.compile()

print("Linear graph compiled successfully.")

Linear graph compiled successfully.


In [6]:
input_data = {
    "question": "Why is LangGraph useful for agent workflows?",
    "retries": 0,
    "max_retries": 2,
    "history_log": []
}

linear_result = linear_graph.invoke(input_data)

print(linear_result["final_answer"])
print("\nSTATE LOG")
for item in linear_result["history_log"]:
    print("-", item)

FINAL ANSWER

Question: Why is LangGraph useful for agent workflows?

LangGraph allows developers to build stateful agent workflows using nodes and edges. It provides explicit control over branching, loops, human approval, and persistence.

Evidence used: 4 items.

STATE LOG
- plan node executed
- retrieve node executed
- generate node executed
- format node executed


In [7]:
print("STATE UPDATES")
for update in linear_graph.stream(input_data, stream_mode="updates"):
    print(update)

STATE UPDATES
{'plan': {'plan': ['Understand the question: Why is LangGraph useful for agent workflows?', 'Collect relevant information', 'Generate an answer', 'Check the quality of the answer'], 'history_log': ['plan node executed']}}
{'retrieve': {'evidence': ['LangGraph builds stateful workflows.', 'Nodes perform individual workflow steps.', 'Edges control transitions.', 'Conditional edges support branching and loops.'], 'history_log': ['plan node executed', 'retrieve node executed']}}
{'generate': {'draft': 'Question: Why is LangGraph useful for agent workflows?\n\nLangGraph allows developers to build stateful agent workflows using nodes and edges. It provides explicit control over branching, loops, human approval, and persistence.\n\nEvidence used: 4 items.', 'history_log': ['plan node executed', 'retrieve node executed', 'generate node executed']}}
{'format': {'final_answer': 'FINAL ANSWER\n============\n\nQuestion: Why is LangGraph useful for agent workflows?\n\nLangGraph allows

# Task 3 — Conditional Edges & Cycles

Now we add a **critique node**.

The first critique intentionally gives a score of **60**. Because this is below 80, the graph returns to `generate`.

On the second pass, the score becomes **90**, so the graph moves to human approval.

The retry counter prevents an infinite cycle.


In [8]:
def critique_node(state):
    retries = state.get("retries", 0)

    if retries == 0:
        score = 60
        critique = "The draft needs better structure."
    else:
        score = 90
        critique = "The revised draft is clear and relevant."

    new_retries = retries + 1

    return {
        "score": score,
        "critique": critique,
        "retries": new_retries,
        "history_log": state.get("history_log", []) + [
            f"critique pass {new_retries}: score={score}"
        ]
    }


def route_after_critique(state):
    score = state.get("score", 0)
    retries = state.get("retries", 0)
    max_retries = state.get("max_retries", 2)

    if score >= 80:
        return "human_approval"

    if retries < max_retries:
        return "generate"

    return "human_approval"

print("Critique node and conditional router created.")

Critique node and conditional router created.


## Why is the loop natural in LangGraph?

A retry loop requires explicit state such as the quality score and retry counter, plus controlled routing back to an earlier step. LangGraph represents these transitions directly as graph edges, making cycles easier to build, inspect, and debug than a simple agent loop.


In [9]:
def human_approval_node(state):
    decision = interrupt({
        "message": "Approve this answer before final formatting?",
        "draft": state.get("draft", ""),
        "score": state.get("score", 0),
        "critique": state.get("critique", "")
    })

    return {
        "approval": str(decision).lower().strip(),
        "history_log": state.get("history_log", []) + [
            f"human approval received: {decision}"
        ]
    }


def approval_route(state):
    if state.get("approval") == "approved":
        return "format"

    return END

print("Human approval node created.")

Human approval node created.


In [10]:
builder = StateGraph(ResearchState)

builder.add_node("plan", plan_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("critique", critique_node)
builder.add_node("human_approval", human_approval_node)
builder.add_node("format", format_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "critique")

builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {
        "generate": "generate",
        "human_approval": "human_approval"
    }
)

builder.add_conditional_edges(
    "human_approval",
    approval_route,
    {
        "format": "format",
        END: END
    }
)

builder.add_edge("format", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Complete LangGraph workflow compiled successfully.")

Complete LangGraph workflow compiled successfully.


In [11]:
# Run until the human approval interrupt.

thread_id = "day3-approved-demo"

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

run_input = {
    "question": "Why is explicit state useful in agent workflows?",
    "retries": 0,
    "max_retries": 2,
    "history_log": []
}

paused_result = graph.invoke(run_input, config=config)

print("Graph reached the human approval point.")
print("Thread ID:", thread_id)

snapshot = graph.get_state(config)
print("Next node:", snapshot.next)
print("Score:", snapshot.values.get("score"))
print("Retries:", snapshot.values.get("retries"))
print("Critique:", snapshot.values.get("critique"))

Graph reached the human approval point.
Thread ID: day3-approved-demo
Next node: ('human_approval',)
Score: 90
Retries: 2
Critique: The revised draft is clear and relevant.


# Task 4 — Human-in-the-Loop

The graph is paused before the final formatting step.

A real product should use human approval for high-impact, difficult-to-reverse, regulated, or uncertain actions. Full autonomy is better for low-risk and reversible tasks with strong validation and monitoring.


In [12]:
# Simulate human approval.

approved_result = graph.invoke(
    Command(resume="approved"),
    config=config
)

print("HUMAN DECISION: APPROVED")
print(approved_result["final_answer"])
print("\nEXECUTION LOG")
for item in approved_result["history_log"]:
    print("-", item)

HUMAN DECISION: APPROVED
FINAL ANSWER

Question: Why is explicit state useful in agent workflows?

LangGraph allows developers to build stateful agent workflows using nodes and edges. It provides explicit control over branching, loops, human approval, and persistence.

Evidence used: 4 items.

EXECUTION LOG
- plan node executed
- retrieve node executed
- generate node executed
- critique pass 1: score=60
- generate node executed
- critique pass 2: score=90
- human approval received: approved
- format node executed


## Rejection Demonstration

We use a separate thread. If the human rejects the answer, the graph ends without creating the final formatted answer.


In [13]:
reject_config = {
    "configurable": {
        "thread_id": "day3-rejected-demo"
    }
}

graph.invoke(
    {
        "question": "What is a conditional edge in LangGraph?",
        "retries": 0,
        "max_retries": 2,
        "history_log": []
    },
    config=reject_config
)

rejected_result = graph.invoke(
    Command(resume="rejected"),
    config=reject_config
)

print("HUMAN DECISION: REJECTED")
print("Approval:", rejected_result.get("approval"))
print("Final answer:", rejected_result.get("final_answer"))
print("\nEXECUTION LOG")
for item in rejected_result["history_log"]:
    print("-", item)

HUMAN DECISION: REJECTED
Approval: rejected
Final answer: None

EXECUTION LOG
- plan node executed
- retrieve node executed
- generate node executed
- critique pass 1: score=60
- generate node executed
- critique pass 2: score=90
- human approval received: rejected


# Task 5 — Persistence & Debugging

`MemorySaver` stores checkpoints in memory. The `thread_id` identifies the workflow state.

This demonstrates persistence during the notebook session. For a production system, use a durable database-backed checkpointer if state must survive application restarts.


In [14]:
# Inspect the saved state of the approved run.

saved_state = graph.get_state(config)

print("SAVED STATE")
print("Next:", saved_state.next)
print("Question:", saved_state.values.get("question"))
print("Score:", saved_state.values.get("score"))
print("Retries:", saved_state.values.get("retries"))
print("Approval:", saved_state.values.get("approval"))

SAVED STATE
Next: ()
Question: Why is explicit state useful in agent workflows?
Score: 90
Retries: 2
Approval: approved


In [15]:
# Inspect checkpoint/state history.

history = list(graph.get_state_history(config))

print("Number of saved snapshots:", len(history))

for i, snapshot in enumerate(history, start=1):
    print(
        f"Snapshot {i}: "
        f"next={snapshot.next}, "
        f"retries={snapshot.values.get('retries')}, "
        f"score={snapshot.values.get('score')}"
    )

Number of saved snapshots: 10
Snapshot 1: next=(), retries=2, score=90
Snapshot 2: next=('format',), retries=2, score=90
Snapshot 3: next=('human_approval',), retries=2, score=90
Snapshot 4: next=('critique',), retries=1, score=60
Snapshot 5: next=('generate',), retries=1, score=60
Snapshot 6: next=('critique',), retries=0, score=None
Snapshot 7: next=('generate',), retries=0, score=None
Snapshot 8: next=('retrieve',), retries=0, score=None
Snapshot 9: next=('plan',), retries=0, score=None
Snapshot 10: next=('__start__',), retries=None, score=None


In [16]:
# Debug one historical snapshot.

if history:
    debug_snapshot = history[4]

    print("DEBUG SNAPSHOT")
    print("Next:", debug_snapshot.next)
    print("Question:", debug_snapshot.values.get("question"))
    print("Score:", debug_snapshot.values.get("score"))
    print("Retries:", debug_snapshot.values.get("retries"))
    print("Critique:", debug_snapshot.values.get("critique"))
    print("Draft:")
    print(debug_snapshot.values.get("draft"))

DEBUG SNAPSHOT
Next: ('generate',)
Question: Why is explicit state useful in agent workflows?
Score: 60
Retries: 1
Critique: The draft needs better structure.
Draft:
Question: Why is explicit state useful in agent workflows?

LangGraph allows developers to build stateful agent workflows using nodes and edges. It provides explicit control over branching, loops, human approval, and persistence.

Evidence used: 4 items.


# Final Workflow — Mermaid Diagram

Copy this into a Markdown cell if your Jupyter environment supports Mermaid:

```mermaid
flowchart TD
    A([START]) --> B[Plan]
    B --> C[Retrieve]
    C --> D[Generate]
    D --> E[Critique]

    E -->|Low Score + Retries Available| D
    E -->|Good Score| F[Human Approval]
    E -->|Retry Limit Reached| F

    F -->|Approved| G[Format]
    F -->|Rejected| H([END])

    G --> I([END])
```

## Final Workflow

**START → Plan → Retrieve → Generate → Critique**

- Low score → Generate again
- Good score → Human Approval
- Approved → Format → END
- Rejected → END


# AgentExecutor vs LangGraph

| Feature | AgentExecutor | LangGraph |
|---|---|---|
| Simple tool agent | Good | Good |
| Explicit state | Limited | Excellent |
| Branching | Possible | Excellent |
| Cycles | Less natural | Natural |
| Human interrupts | More custom | Built-in |
| Checkpointing | More custom | Built-in |
| Debugging/history | Limited | Strong |
| Complex workflows | Less suitable | Excellent |

### Conclusion

Use **AgentExecutor** for a relatively simple tool-using agent. Use **LangGraph** when you need explicit state, branching, retry loops, human approval, persistence, and detailed workflow control.
